# Split Point-Cloud Viewer (diagnostic tool)

Compare **two point clouds side by side** in a desktop (tkinter) window: five spatial
views each, per-field statistics, colouring by any data field, and value/range
filtering. Built to answer questions like *"which fields should be merged?"* and
*"how do the class labels of dataset A map onto dataset B?"* — groundwork for the
planned `class_map` reclassifier in `pipeline/forainet_prep.py`.

**Supported formats**

| Format | How fields are found |
|---|---|
| `.las` / `.laz` | all point dimensions (incl. extra dims like `tree_ID`, `dist_axes`, `Class`); `x/y/z` are the scaled coordinates |
| `.ply` | all vertex properties |
| `.npy` (structured array) | names taken from the array's dtype |
| `.npy` (plain N×K matrix) | names taken from a **JSON sidecar** (see below) |

**The `.npy` JSON sidecar convention**

A bare `.npy` matrix stores no column names, so the viewer looks for a JSON file with
the **same stem** next to it — `plot_01_007.npy` → `plot_01_007.json`:

```json
{"columns": ["x", "y", "z", "2tree_ID"]}
```

Any *additional* keys in that JSON are treated as metadata about the cloud and shown
in the info box. If the sidecar is missing, unreadable, or its `columns` count does
not match the matrix, the viewer falls back to `x, y, z, col_3, …` and shows a
warning. (The sidecar files will be produced by a separate export script in a later
stage.)

**How to run**

1. Use the **aifor** kernel (`mamba env` with `laspy`, `plyfile`, `matplotlib`).
2. Run the cells top to bottom; the last cell opens the viewer window.
3. ⚠️ The notebook kernel is **busy while the window is open** (tkinter's event loop
   runs in the kernel). Close the window to get the kernel back.

In [1]:
"""Imports.

matplotlib is embedded directly into the tkinter window via ``FigureCanvasTkAgg``:
we build ``Figure`` objects ourselves and never touch ``pyplot`` / ``plt.show()``.
Keeping pyplot (and its hidden global state) out of the picture is the recommended
way to host matplotlib inside a GUI toolkit.
"""
import json
import tkinter as tk
from pathlib import Path
from tkinter import filedialog, messagebox, ttk
from tkinter.scrolledtext import ScrolledText

import laspy
import numpy as np
import pandas as pd
from matplotlib import colormaps
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from plyfile import PlyData

print("imports OK")

imports OK


In [2]:
# ---------------------------------------------------------------------------
# Loaders: every supported format is normalised into ONE structure so the GUI
# (and any code you write in later cells) never has to care where a cloud
# came from:
#
#   {
#     "name":       "plot_01_tree_ID_dist_axes.las",   # file name (display)
#     "path":       Path(...),                          # full path
#     "file_type":  "LAS/LAZ" | "PLY" | "NPY",
#     "num_points": 18_123_456,
#     "fields":     {"x": ndarray, "y": ndarray, ...},  # plain 1-D numpy arrays
#     "meta":       {...},   # extra sidecar keys (NPY only), shown in info box
#     "warnings":   [...],   # human-readable problems, shown in info box
#   }
#
# laspy gotcha (repo convention): laspy exposes x/y/z as ScaledArrayView and
# bit fields (return_number, ...) as SubFieldView. Those are NOT plain arrays
# and confuse numpy/pandas, so every accessor is materialised with np.asarray.
# ---------------------------------------------------------------------------

# Raw integer coordinate dimensions of the LAS spec. We drop them and expose the
# scaled x/y/z instead (raw ints are scale*X+offset away from real coordinates).
_LAS_RAW_COORDS = ("X", "Y", "Z")


def _load_las(path: Path) -> dict:
    """Read a .las/.laz file into the common cloud structure."""
    las = laspy.read(str(path))
    fields = {
        "x": np.asarray(las.x, dtype=np.float64),
        "y": np.asarray(las.y, dtype=np.float64),
        "z": np.asarray(las.z, dtype=np.float64),
    }
    # Every remaining dimension, including extra-bytes dims (tree_ID, dist_axes,
    # Class, ...) that 3DFin / forainet_prep attach.
    for dim in las.point_format.dimension_names:
        if dim in _LAS_RAW_COORDS:
            continue
        fields[dim] = np.asarray(getattr(las, dim))
    return {
        "name": path.name,
        "path": path,
        "file_type": "LAS/LAZ",
        "num_points": len(fields["x"]),
        "fields": fields,
        "meta": {},
        "warnings": [],
    }


def _load_ply(path: Path) -> dict:
    """Read a .ply file (all vertex properties) into the common cloud structure."""
    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    fields = {name: np.asarray(vertex[name]) for name in vertex.dtype.names}
    return {
        "name": path.name,
        "path": path,
        "file_type": "PLY",
        "num_points": len(vertex),
        "fields": fields,
        "meta": {},
        "warnings": [],
    }


def _npy_column_names(path: Path, n_cols: int) -> tuple:
    """Resolve column names for a bare N x K .npy matrix.

    Looks for a JSON sidecar with the same stem (plot_01_007.npy ->
    plot_01_007.json) holding at least {"columns": [...]}. Every OTHER key of
    that JSON is returned as metadata so the GUI can display it. Falls back to
    generic names (x, y, z, col_3, ...) whenever the sidecar is unusable, and
    reports what happened through the warnings list.

    Returns (names, meta, warnings).
    """
    warnings, meta = [], {}
    names = None

    sidecar = path.with_suffix(".json")
    if sidecar.exists():
        try:
            info = json.loads(sidecar.read_text(encoding="utf-8"))
            cols = info.get("columns")
            if isinstance(cols, list) and len(cols) == n_cols:
                names = [str(c) for c in cols]
                meta = {k: v for k, v in info.items() if k != "columns"}
            else:
                got = len(cols) if isinstance(cols, list) else repr(cols)
                warnings.append(
                    f"sidecar {sidecar.name}: 'columns' does not match the array "
                    f"({got} names vs {n_cols} columns)"
                )
        except (OSError, json.JSONDecodeError) as exc:
            warnings.append(f"sidecar {sidecar.name} could not be read: {exc}")
    else:
        warnings.append(f"no sidecar {sidecar.name} next to the array")

    if names is None:
        names = ["x", "y", "z"][:n_cols] + [f"col_{i}" for i in range(3, n_cols)]
        warnings.append("using fallback column names: " + ", ".join(names))
    return names, meta, warnings


def _load_npy(path: Path) -> dict:
    """Read a .npy file (structured array or bare N x K matrix)."""
    arr = np.load(str(path), allow_pickle=False)
    meta, warnings = {}, []

    if arr.dtype.names:
        # Structured array: numpy already stored the column names for us.
        fields = {name: np.asarray(arr[name]) for name in arr.dtype.names}
    else:
        if arr.ndim == 1:  # a single column still gets the full treatment
            arr = arr.reshape(-1, 1)
        if arr.ndim != 2:
            raise ValueError(f"{path.name}: expected an N x K matrix, got shape {arr.shape}")
        names, meta, warnings = _npy_column_names(path, arr.shape[1])
        # ascontiguousarray: column slices of a 2-D array are strided views;
        # copying them keeps all downstream numpy ops fast and predictable.
        fields = {name: np.ascontiguousarray(arr[:, i]) for i, name in enumerate(names)}

    n = len(next(iter(fields.values())))
    return {
        "name": path.name,
        "path": path,
        "file_type": "NPY",
        "num_points": n,
        "fields": fields,
        "meta": meta,
        "warnings": warnings,
    }


_LOADERS = {".las": _load_las, ".laz": _load_las, ".ply": _load_ply, ".npy": _load_npy}


def load_point_cloud(path) -> dict:
    """Load any supported point-cloud file into the common structure."""
    path = Path(path)
    loader = _LOADERS.get(path.suffix.lower())
    if loader is None:
        raise ValueError(
            f"unsupported file type '{path.suffix}' "
            f"(supported: {', '.join(sorted(_LOADERS))})"
        )
    return loader(path)


def summarize_fields(cloud: dict) -> pd.DataFrame:
    """Per-field statistics table: dtype, Min, Max, Unique.

    Unique counts are only computed for integer fields (labels, ids): they are
    the interesting ones for label diagnosis, and np.unique on huge float
    columns would be slow for no benefit.
    """
    rows = []
    for name, col in cloud["fields"].items():
        row = {"Field": name, "dtype": str(col.dtype), "Min": "-", "Max": "-", "Unique": "-"}
        if np.issubdtype(col.dtype, np.number):
            row["Min"] = f"{col.min():g}"
            row["Max"] = f"{col.max():g}"
            if np.issubdtype(col.dtype, np.integer):
                row["Unique"] = f"{np.unique(col).size:,}"
        rows.append(row)
    return pd.DataFrame(rows)


def cloud_info_text(cloud: dict) -> str:
    """The full plain-text report shown in a panel's info box."""
    lines = [
        f"{cloud['file_type']}  |  {cloud['num_points']:,} points",
        str(cloud["path"]),
        "",
        summarize_fields(cloud).to_string(index=False),
    ]
    if cloud["meta"]:
        lines += ["", "Sidecar metadata:"]
        lines += [f"  {k}: {v}" for k, v in cloud["meta"].items()]
    if cloud["warnings"]:
        lines += ["", "WARNINGS:"]
        lines += [f"  ! {w}" for w in cloud["warnings"]]
    return "\n".join(lines)


print("loaders defined")

loaders defined


In [3]:
# ---------------------------------------------------------------------------
# Filtering + projection helpers.
#
# These are pure functions (no GUI, no globals) on purpose: the tkinter class
# below only wires widgets to them, and they can be reused / unit-tested from
# plain notebook cells.
# ---------------------------------------------------------------------------


def parse_values(text: str) -> list:
    """Parse a comma-separated values string ("2, 3, 4.5") into numbers.

    Each item is tried as int first, then float, else kept as a string (useful
    should a field ever hold non-numeric data). Empty items are skipped.
    """
    values = []
    for item in text.split(","):
        item = item.strip()
        if not item:
            continue
        try:
            values.append(int(item))
        except ValueError:
            try:
                values.append(float(item))
            except ValueError:
                values.append(item)
    return values


def make_mask(fields: dict, field: str, mode: str,
              vmin=None, vmax=None, values=None) -> np.ndarray:
    """Boolean keep-mask over all points of a cloud.

    mode="range":  keep points with vmin <= field <= vmax
                   (either bound may be None = unbounded on that side).
    mode="values": keep points whose field is in `values` (exact match),
                   e.g. Class in {2, 3}.
    """
    col = fields[field]
    if mode == "range":
        mask = np.ones(len(col), dtype=bool)
        if vmin is not None:
            mask &= col >= vmin
        if vmax is not None:
            mask &= col <= vmax
        return mask
    if mode == "values":
        if not values:
            return np.ones(len(col), dtype=bool)
        return np.isin(col, values)
    raise ValueError(f"unknown filter mode: {mode!r}")


# The five spatial views. The isometric ones rotate the cloud about the z axis
# by the given angle and then look at it from the front (rotated-x vs z), i.e.
# a "camera walking around the plot" at 45 deg steps between the axis-aligned
# front/side views.
VIEWS = (
    ("XY (top)", "xy"),
    ("XZ (front)", "xz"),
    ("YZ (side)", "yz"),
    ("ISO 45\N{DEGREE SIGN}", "iso45"),
    ("ISO 135\N{DEGREE SIGN}", "iso135"),
)


def project(x: np.ndarray, y: np.ndarray, z: np.ndarray, view: str):
    """Project 3-D points onto the 2-D plane of the requested view.

    Returns (u, v): the horizontal and vertical plot coordinates.
    """
    if view == "xy":
        return x, y
    if view == "xz":
        return x, z
    if view == "yz":
        return y, z
    if view in ("iso45", "iso135"):
        theta = np.deg2rad(45.0 if view == "iso45" else 135.0)
        return x * np.cos(theta) + y * np.sin(theta), z
    raise ValueError(f"unknown view: {view!r}")


print("helpers defined")

helpers defined


In [4]:
# ---------------------------------------------------------------------------
# The tkinter GUI.
#
# Layout: one window, two identical panels (Cloud A | Cloud B). Each panel is
# fully independent -- its own file, its own colour field, its own filter --
# because the whole point is comparing two datasets whose fields DIFFER.
#
#   +------------------------- Cloud A -------------------------+  (same for B)
#   | [Load...]  plot_01_tree_ID_dist_axes.las                  |
#   | info box: type, #points, per-field dtype/min/max/unique   |
#   | Colour by: [Class v]   Max points: [100000]   [Redraw]    |
#   | Filter: [tree_ID v] (o) Range ( ) Values                  |
#   |         min [   ] max [   ] values [2, 3]  [Apply][Reset] |
#   | shown 100,000 / filtered 3,412,111 / total 18,000,000     |
#   |  [XY (top)] [XZ (front)] [YZ (side)]                      |
#   |  [ISO 45]   [ISO 135]    [legend / colourbar]             |
#   |  (matplotlib toolbar: zoom / pan / save PNG)              |
#   +------------------------------------------------------------+
# ---------------------------------------------------------------------------

# Fields tried (in order) as the default colour/filter field after loading --
# the label-ish fields are what this tool is usually used to inspect.
_PREFERRED_FIELDS = ("Class", "semantic_seg", "classification",
                     "tree_ID", "treeID", "2tree_ID")
_MAX_LEGEND_CLASSES = 20     # <= this many unique ints -> discrete colours + legend
_DEFAULT_MAX_POINTS = 100_000
_DOWNSAMPLE_SEED = 42        # fixed seed -> the same subsample on every redraw
_FILE_TYPES = [("Point clouds", "*.las *.laz *.ply *.npy"), ("All files", "*.*")]


class CloudPanel:
    """One half of the window: load / inspect / filter / draw a single cloud."""

    def __init__(self, parent, title):
        self.cloud = None      # the loaded cloud dict (see load_point_cloud)
        self.mask = None       # bool keep-mask from the filter (None = no filter)
        self._legend = None    # legend artist, replaced on every redraw

        self.frame = ttk.LabelFrame(parent, text=title, padding=4)

        # -- row: load button + file name ----------------------------------
        row = ttk.Frame(self.frame)
        row.pack(fill="x")
        ttk.Button(row, text="Load\N{HORIZONTAL ELLIPSIS}", command=self.load_file).pack(side="left")
        self.file_label = ttk.Label(row, text="(no file loaded)", anchor="w")
        self.file_label.pack(side="left", fill="x", expand=True, padx=6)

        # -- info box -------------------------------------------------------
        self.info = ScrolledText(self.frame, height=9, width=40, font=("Consolas", 8),
                                 state="disabled", wrap="none")
        self.info.pack(fill="x", pady=(4, 4))

        # -- row: colour-by field, max points, redraw -----------------------
        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="Colour by:").pack(side="left")
        self.color_var = tk.StringVar()
        self.color_combo = ttk.Combobox(row, textvariable=self.color_var,
                                        state="readonly", width=16)
        self.color_combo.pack(side="left", padx=(2, 10))
        self.color_combo.bind("<<ComboboxSelected>>", lambda _e: self.redraw())
        ttk.Label(row, text="Max points:").pack(side="left")
        self.max_points_var = tk.StringVar(value=str(_DEFAULT_MAX_POINTS))
        ttk.Entry(row, textvariable=self.max_points_var, width=9).pack(side="left", padx=(2, 10))
        ttk.Button(row, text="Redraw", command=self.redraw).pack(side="left")

        # -- rows: the filter -----------------------------------------------
        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="Filter:").pack(side="left")
        self.filter_var = tk.StringVar()
        self.filter_combo = ttk.Combobox(row, textvariable=self.filter_var,
                                         state="readonly", width=16)
        self.filter_combo.pack(side="left", padx=(2, 10))
        self.mode_var = tk.StringVar(value="range")
        ttk.Radiobutton(row, text="Range", variable=self.mode_var,
                        value="range").pack(side="left")
        ttk.Radiobutton(row, text="Values", variable=self.mode_var,
                        value="values").pack(side="left")

        row = ttk.Frame(self.frame)
        row.pack(fill="x", pady=(0, 2))
        ttk.Label(row, text="min").pack(side="left")
        self.min_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.min_var, width=9).pack(side="left", padx=(2, 6))
        ttk.Label(row, text="max").pack(side="left")
        self.max_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.max_var, width=9).pack(side="left", padx=(2, 6))
        ttk.Label(row, text="values").pack(side="left")
        self.values_var = tk.StringVar()
        ttk.Entry(row, textvariable=self.values_var, width=14).pack(side="left", padx=(2, 6))
        ttk.Button(row, text="Apply", command=self.apply_filter).pack(side="left", padx=(4, 2))
        ttk.Button(row, text="Reset", command=self.reset_filter).pack(side="left")

        self.status = ttk.Label(self.frame, text="", anchor="w", foreground="#555")
        self.status.pack(fill="x", pady=(0, 2))

        # -- the figure: 2 x 3 grid = 5 views + legend/colourbar slot -------
        self.fig = Figure(figsize=(7.2, 6.2), dpi=100)
        self.axes = [self.fig.add_subplot(2, 3, i + 1) for i in range(5)]
        self.ax_extra = self.fig.add_subplot(2, 3, 6)
        self.ax_extra.axis("off")
        # The colourbar axes is created ONCE and only toggled visible/hidden:
        # destroying and recreating it on every redraw trips matplotlib's
        # Colorbar.remove() (it assumes gridspec-managed axes, ours is an inset).
        self.cax = self.ax_extra.inset_axes([0.35, 0.08, 0.14, 0.84])
        self.cax.set_visible(False)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        # The standard matplotlib toolbar: zoom box, pan, back/forward, save PNG.
        toolbar_frame = ttk.Frame(self.frame)
        toolbar_frame.pack(fill="x")
        NavigationToolbar2Tk(self.canvas, toolbar_frame)

    # -- loading ------------------------------------------------------------

    def load_file(self):
        path = filedialog.askopenfilename(title="Select a point cloud",
                                          filetypes=_FILE_TYPES)
        if not path:  # dialog cancelled
            return
        try:
            cloud = load_point_cloud(path)
        except Exception as exc:  # bad file, unsupported layout, ...
            messagebox.showerror("Load failed", f"{path}\n\n{exc}")
            return
        self.cloud = cloud
        self.mask = None
        self.file_label.config(text=cloud["name"])
        self._set_info(cloud_info_text(cloud))

        # Populate both field selectors; default to the first label-ish field.
        names = list(cloud["fields"])
        self.color_combo["values"] = names
        self.filter_combo["values"] = names
        default = next((f for f in _PREFERRED_FIELDS if f in cloud["fields"]),
                       names[0] if names else "")
        self.color_var.set(default)
        self.filter_var.set(default)
        self.redraw()

    def _set_info(self, text):
        self.info.config(state="normal")
        self.info.delete("1.0", "end")
        self.info.insert("1.0", text)
        self.info.config(state="disabled")

    # -- filtering ----------------------------------------------------------

    def apply_filter(self):
        if self.cloud is None:
            return
        field = self.filter_var.get()
        if field not in self.cloud["fields"]:
            messagebox.showwarning("Filter", f"Unknown field: {field!r}")
            return
        mode = self.mode_var.get()
        try:
            if mode == "range":
                vmin = float(self.min_var.get()) if self.min_var.get().strip() else None
                vmax = float(self.max_var.get()) if self.max_var.get().strip() else None
                self.mask = make_mask(self.cloud["fields"], field, "range",
                                      vmin=vmin, vmax=vmax)
            else:
                values = parse_values(self.values_var.get())
                if not values:
                    messagebox.showwarning("Filter", "Enter comma-separated values, e.g. 2, 3")
                    return
                self.mask = make_mask(self.cloud["fields"], field, "values", values=values)
        except ValueError as exc:
            messagebox.showerror("Filter", str(exc))
            return
        self.redraw()

    def reset_filter(self):
        self.mask = None
        self.redraw()

    # -- drawing ------------------------------------------------------------

    def _max_points(self) -> int:
        try:
            return max(1, int(self.max_points_var.get()))
        except ValueError:
            self.max_points_var.set(str(_DEFAULT_MAX_POINTS))
            return _DEFAULT_MAX_POINTS

    def redraw(self):
        if self.cloud is None:
            return
        fields = self.cloud["fields"]
        missing = [c for c in ("x", "y", "z") if c not in fields]
        if missing:
            self.status.config(text=f"cannot draw: missing coordinate field(s) {missing}")
            return

        total = self.cloud["num_points"]
        keep = self.mask if self.mask is not None else np.ones(total, dtype=bool)
        n_filtered = int(keep.sum())

        # Random downsample AFTER filtering so the view stays responsive on
        # multi-million-point plots. Fixed seed = stable picture across redraws.
        idx = np.flatnonzero(keep)
        max_pts = self._max_points()
        if idx.size > max_pts:
            rng = np.random.default_rng(_DOWNSAMPLE_SEED)
            idx = rng.choice(idx, size=max_pts, replace=False)

        x, y, z = fields["x"][idx], fields["y"][idx], fields["z"][idx]

        # -- colours ---------------------------------------------------------
        color_field = self.color_var.get()
        cvals = fields.get(color_field)
        cvals = cvals[idx] if cvals is not None else z  # sensible fallback
        discrete = (np.issubdtype(cvals.dtype, np.integer)
                    and np.unique(cvals).size <= _MAX_LEGEND_CLASSES)
        if discrete:
            uniq = np.unique(cvals)
            tab20 = colormaps["tab20"]
            palette = np.array([tab20(i % 20) for i in range(uniq.size)])
            point_colors = palette[np.searchsorted(uniq, cvals)]
            scatter_kw = dict(c=point_colors)
        else:
            scatter_kw = dict(c=cvals.astype(np.float64), cmap="viridis")

        # -- the five views ---------------------------------------------------
        last_sc = None
        for ax, (label, view) in zip(self.axes, VIEWS):
            ax.clear()
            u, v = project(x, y, z, view)
            last_sc = ax.scatter(u, v, s=1.5, linewidths=0, **scatter_kw)
            ax.set_title(label, fontsize=9)
            ax.tick_params(labelsize=7)
            ax.set_aspect("equal", adjustable="datalim")

        # -- legend / colourbar in the 6th slot -------------------------------
        if self._legend is not None:   # drop last redraw's legend
            self._legend.remove()
            self._legend = None
        self.cax.clear()
        if discrete:
            self.cax.set_visible(False)
            handles = [Line2D([], [], marker="o", linestyle="", markersize=6,
                              markerfacecolor=palette[i], markeredgecolor="none",
                              label=str(val))
                       for i, val in enumerate(uniq)]
            self._legend = self.ax_extra.legend(
                handles=handles, loc="center", fontsize=8,
                title=color_field, title_fontsize=9,
                ncols=1 + uniq.size // 11, frameon=False)
        elif last_sc is not None:
            self.cax.set_visible(True)
            self.fig.colorbar(last_sc, cax=self.cax)
            self.cax.set_title(color_field, fontsize=8)
            self.cax.tick_params(labelsize=7)

        self.fig.tight_layout()
        self.canvas.draw_idle()
        self.status.config(
            text=f"shown {idx.size:,} / filtered {n_filtered:,} / total {total:,}"
                 + ("" if self.mask is None else "   [filter active]")
        )


class PointCloudCompareApp:
    """The main window: two independent CloudPanels side by side."""

    def __init__(self, root):
        root.title("Split Point-Cloud Viewer \N{EM DASH} 2-cloud diagnostic")
        root.geometry("1760x1000")
        container = ttk.Frame(root, padding=4)
        container.pack(fill="both", expand=True)
        self.left = CloudPanel(container, "Cloud A")
        self.right = CloudPanel(container, "Cloud B")
        self.left.frame.pack(side="left", fill="both", expand=True, padx=(0, 3))
        self.right.frame.pack(side="left", fill="both", expand=True, padx=(3, 0))


def launch_viewer(auto_close_ms=None):
    """Open the viewer window. Blocks until the window is closed.

    auto_close_ms: close the window automatically after N milliseconds --
    only used by automated smoke tests, leave as None for normal use.
    """
    root = tk.Tk()
    app = PointCloudCompareApp(root)
    if auto_close_ms is not None:
        root.after(auto_close_ms, root.destroy)
    root.mainloop()
    return app


print("GUI defined")

GUI defined


---
## Launch the viewer

Running the next cell opens the desktop window. **The kernel stays busy until you
close the window** — that is normal for a tkinter GUI inside a notebook.

Tips:
- Load a different file at any time with **Load…** (each panel is independent).
- The **matplotlib toolbar** under each figure gives zoom / pan / save-as-PNG.
- **Filter → Values** with e.g. `2, 3` on a `Class` field shows only those labels;
  **Range** with min/max works better for continuous fields (`z`, `dist_axes`).
- Raise **Max points** for more detail (slower), lower it for speed.

In [5]:
# Opens the viewer window; the kernel is busy until the window is closed.
app = launch_viewer()